In [ ]:
# =========================
# Colab Script: Best-model rerun for paired t-test (O vs OF)
# - ONLY LightGBM
# - Compare: O (orig only) vs OF (orig + top-K edges) for GES, K=34
# - Run N_RUNS with different seeds -> paired t-test
# - Safe Google Drive mount (no "mountpoint contains files" crash)
# =========================

!pip -q install lightgbm scipy

import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from scipy.stats import ttest_rel, wilcoxon

# -------------------------
# SAFE Drive mount
# -------------------------
from google.colab import drive
if os.path.exists("/content/drive/MyDrive/colab_projects"):
    print("[OK] Drive already mounted.")
else:
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount("/content/drive", force_remount=False)

# =========================
# Logging helper
# =========================
T0 = time.time()
def log(msg: str):
    dt = time.time() - T0
    print(f"[{dt:8.1f}s] {msg}")

# =========================
# Warning / Log control
# =========================
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    lgb.register_logger(lambda msg: None)
except Exception:
    pass

# =========================
# Config
# =========================
RANDOM_STATE = 42
K_EDGE = 34
DAG = "GES"

N_RUNS = 30
SEEDS = [RANDOM_STATE + i for i in range(N_RUNS)]

LGBM_CPU_PARAMS_BASE = dict(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    verbose=-1,
)

BASE_DIR = "/content/drive/MyDrive/bank_failure_prediction"
OUT_DIR = os.path.join(BASE_DIR, "results_tables_ttest")
os.makedirs(OUT_DIR, exist_ok=True)

DATA_BASE = {
    "train": os.path.join(BASE_DIR, "data_with_features_train.csv"),
    "val":   os.path.join(BASE_DIR, "data_with_features_val.csv"),
    "test":  os.path.join(BASE_DIR, "data_with_features_test.csv"),
}

DATA_GES = {
    "train": os.path.join(BASE_DIR, "data_with_features_GES_train.csv"),
    "val":   os.path.join(BASE_DIR, "data_with_features_GES_val.csv"),
    "test":  os.path.join(BASE_DIR, "data_with_features_GES_test.csv"),
}

# =========================
# Seed utilities
# =========================
UINT32_MAX = 2**32 - 1
def seed_uint32(x: int) -> int:
    return int(x) % UINT32_MAX

def set_seed_fixed(seed: int):
    s = seed_uint32(seed)
    random.seed(s)
    np.random.seed(s)

# =========================
# HARD BLOCK: index-like / Unnamed remover
# =========================
def is_index_col_name(col: str) -> bool:
    c = str(col).strip()
    cl = c.lower()
    if c.startswith("Unnamed") or cl.startswith("unnamed"):
        return True
    if cl in {"index", "_index"}:
        return True
    if cl.endswith("_index"):
        return True
    return False

def looks_like_index_series(s: pd.Series) -> bool:
    try:
        v = pd.to_numeric(s, errors="coerce")
        if v.isna().mean() > 0.3:
            return False
        n = len(v)
        if n <= 5:
            return False

        uniq_ratio = v.nunique(dropna=True) / float(n)
        if uniq_ratio < 0.98:
            return False

        vv = v.to_numpy()
        if np.all(vv == np.arange(n)):
            return True
        if np.all(vv == np.arange(1, n + 1)):
            return True

        if np.all(np.diff(vv) >= 0):
            dif = np.diff(vv)
            if np.mean(np.abs(dif - 1.0) < 1e-9) > 0.95:
                return True
    except Exception:
        return False
    return False

def drop_all_indexlike_cols(df: pd.DataFrame, name: str, aggressive_value_check: bool = True) -> pd.DataFrame:
    if df is None or df.shape[1] == 0:
        return df

    drop_cols = []
    for c in list(df.columns):
        if is_index_col_name(c):
            drop_cols.append(c)

    if aggressive_value_check:
        for c in list(df.columns):
            if c in drop_cols:
                continue
            try:
                if looks_like_index_series(df[c]):
                    drop_cols.append(c)
            except Exception:
                pass

    if drop_cols:
        drop_cols = list(dict.fromkeys(drop_cols))
        log(f"[DROP] {name}: {drop_cols}")
        df = df.drop(columns=drop_cols)

    bad = [c for c in df.columns if str(c).lower().startswith("unnamed")]
    if bad:
        raise RuntimeError(f"[FATAL] {name}: Unnamed still exists: {bad}")

    return df

def filter_out_index_cols(cols):
    out = []
    for c in cols:
        if is_index_col_name(c):
            continue
        if str(c).strip().lower().startswith("unnamed"):
            continue
        out.append(c)
    return out

def read_csv_safely(path: str, name: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = drop_all_indexlike_cols(df, name=name, aggressive_value_check=True)
    return df

# =========================
# Data utils
# =========================
def detect_target_col(df: pd.DataFrame) -> str:
    candidates = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found. Tried {candidates}")

def get_original_cols(df_any: pd.DataFrame, target_col: str):
    cols = []
    for c in df_any.columns:
        if c == target_col:
            continue
        if str(c).startswith("edge_"):
            continue
        if is_index_col_name(c):
            continue
        if str(c).strip().lower().startswith("unnamed"):
            continue
        cols.append(c)
    return filter_out_index_cols(cols)

def ensure_cols(df: pd.DataFrame, cols, name: str):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise ValueError(f"[{name}] Missing columns: {miss[:30]} (total {len(miss)})")

def to_numpy32(df: pd.DataFrame, cols, name: str = "") -> np.ndarray:
    cols = filter_out_index_cols(cols)
    if any(str(c).lower().startswith("unnamed") for c in cols):
        raise RuntimeError(f"[FATAL] {name}: Unnamed in cols.")
    return df[cols].to_numpy(dtype=np.float32)

# =========================
# Edge ranking by train std
# =========================
def get_edge_cols(df_alg: pd.DataFrame, alg: str):
    prefix = f"edge_{alg}__"
    cols = [c for c in df_alg.columns if str(c).startswith(prefix)]
    if len(cols) == 0:
        cols = [c for c in df_alg.columns if str(c).startswith("edge_")]
    return filter_out_index_cols(cols)

def rank_edges_by_train_std(df_alg_train: pd.DataFrame, edge_cols):
    edge_cols = filter_out_index_cols(edge_cols)
    if len(edge_cols) == 0:
        return []
    stds = (
        df_alg_train[edge_cols]
        .astype(float)
        .std(axis=0)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )
    return stds.sort_values(ascending=False).index.tolist()

# =========================
# LightGBM runner
# =========================
def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

def run_lgbm_once(seed: int, X_tr, y_tr, X_va, y_va, X_te, y_te):
    s = seed_uint32(seed)
    params = dict(LGBM_CPU_PARAMS_BASE)
    params["random_state"] = s

    clf = lgb.LGBMClassifier(**params)
    clf.fit(X_tr, y_tr)

    va_prob = clf.predict_proba(X_va)[:, 1]
    thr = best_f1_threshold(y_va, va_prob)

    te_prob = clf.predict_proba(X_te)[:, 1]
    te_pred = (te_prob >= thr).astype(int)

    return {
        "AUROC": float(roc_auc_score(y_te, te_prob)),
        "AUPRC": float(average_precision_score(y_te, te_prob)),
        "F1": float(f1_score(y_te, te_pred, zero_division=0)),
        "THR": float(thr),
    }

# =========================
# 0) Check files
# =========================
log("checking files...")
for k in ["train", "val", "test"]:
    if not os.path.exists(DATA_BASE[k]):
        raise FileNotFoundError(f"Missing base {k}: {DATA_BASE[k]}")
    if not os.path.exists(DATA_GES[k]):
        raise FileNotFoundError(f"Missing GES {k}: {DATA_GES[k]}")

# =========================
# 1) Load BASE (O)
# =========================
log("loading BASE...")
df_base_train = read_csv_safely(DATA_BASE["train"], "BASE_train")
df_base_val   = read_csv_safely(DATA_BASE["val"],   "BASE_val")
df_base_test  = read_csv_safely(DATA_BASE["test"],  "BASE_test")

target_col = detect_target_col(df_base_train)
orig_cols = get_original_cols(df_base_train, target_col)
log(f"target_col={target_col} | n_orig_cols={len(orig_cols)}")

ensure_cols(df_base_val,  orig_cols + [target_col], "BASE_VAL")
ensure_cols(df_base_test, orig_cols + [target_col], "BASE_TEST")

X_tr_O = to_numpy32(df_base_train, orig_cols, "O_train")
X_va_O = to_numpy32(df_base_val,   orig_cols, "O_val")
X_te_O = to_numpy32(df_base_test,  orig_cols, "O_test")

y_tr = df_base_train[target_col].to_numpy(np.int64)
y_va = df_base_val[target_col].to_numpy(np.int64)
y_te = df_base_test[target_col].to_numpy(np.int64)

# =========================
# 2) Load GES (OF)
# =========================
log("loading GES...")
df_ges_train = read_csv_safely(DATA_GES["train"], "GES_train")
df_ges_val   = read_csv_safely(DATA_GES["val"],   "GES_val")
df_ges_test  = read_csv_safely(DATA_GES["test"],  "GES_test")

ensure_cols(df_ges_train, orig_cols + [target_col], "GES_TRAIN_ORIG")
ensure_cols(df_ges_val,   orig_cols + [target_col], "GES_VAL_ORIG")
ensure_cols(df_ges_test,  orig_cols + [target_col], "GES_TEST_ORIG")

edge_cols_all = get_edge_cols(df_ges_train, "GES")
edge_ranked = rank_edges_by_train_std(df_ges_train, edge_cols_all)

if len(edge_ranked) < K_EDGE:
    raise ValueError(f"[GES] edges(total)={len(edge_ranked)} < K_EDGE={K_EDGE}")

edge_used = filter_out_index_cols(edge_ranked[:K_EDGE])
of_cols = filter_out_index_cols(orig_cols + edge_used)

log(f"OF ready | K_EDGE={K_EDGE} | n_feat_OF={len(of_cols)} (orig={len(orig_cols)} + edge={len(edge_used)})")

X_tr_OF = to_numpy32(df_ges_train, of_cols, "OF_train")
X_va_OF = to_numpy32(df_ges_val,   of_cols, "OF_val")
X_te_OF = to_numpy32(df_ges_test,  of_cols, "OF_test")

# =========================
# 3) Paired repeated runs
# =========================
log(f"running paired repeats... N_RUNS={N_RUNS}")
rows = []
for i, sd in enumerate(SEEDS, 1):
    set_seed_fixed(sd)

    mO  = run_lgbm_once(sd, X_tr_O,  y_tr, X_va_O,  y_va, X_te_O,  y_te)
    mOF = run_lgbm_once(sd, X_tr_OF, y_tr, X_va_OF, y_va, X_te_OF, y_te)

    rows.append({
        "RUN": i,
        "SEED": int(sd),
        "AUROC_O": mO["AUROC"],
        "AUPRC_O": mO["AUPRC"],
        "F1_O": mO["F1"],
        "THR_O": mO["THR"],
        "AUROC_OF": mOF["AUROC"],
        "AUPRC_OF": mOF["AUPRC"],
        "F1_OF": mOF["F1"],
        "THR_OF": mOF["THR"],
        "d_AUROC": mOF["AUROC"] - mO["AUROC"],
        "d_AUPRC": mOF["AUPRC"] - mO["AUPRC"],
        "d_F1": mOF["F1"] - mO["F1"],
    })

    if (i == 1) or (i % 5 == 0) or (i == N_RUNS):
        log(f"  run {i}/{N_RUNS} seed={sd} | d_AUPRC={rows[-1]['d_AUPRC']:+.6f} d_AUROC={rows[-1]['d_AUROC']:+.6f} d_F1={rows[-1]['d_F1']:+.6f}")

df_runs = pd.DataFrame(rows)

# =========================
# 4) Paired tests
# =========================
def summarize_tests(metric: str):
    o  = df_runs[f"{metric}_O"].to_numpy(float)
    of = df_runs[f"{metric}_OF"].to_numpy(float)
    d  = of - o

    t = ttest_rel(of, o)

    try:
        w = wilcoxon(d)
        w_stat, w_p = float(w.statistic), float(w.pvalue)
    except Exception:
        w_stat, w_p = np.nan, np.nan

    return {
        "METRIC": metric,
        "MEAN_O": float(np.mean(o)),
        "STD_O": float(np.std(o, ddof=1)),
        "MEAN_OF": float(np.mean(of)),
        "STD_OF": float(np.std(of, ddof=1)),
        "MEAN_DIFF(OF-O)": float(np.mean(d)),
        "STD_DIFF": float(np.std(d, ddof=1)),
        "T_STAT": float(t.statistic),
        "T_P": float(t.pvalue),
        "WILCOXON_STAT": w_stat,
        "WILCOXON_P": w_p,
    }

summary = pd.DataFrame([
    summarize_tests("AUPRC"),
    summarize_tests("AUROC"),
    summarize_tests("F1"),
])

log("t-test summary:")
display(summary)
display(df_runs.head(10))

# =========================
# 5) Save
# =========================
out_runs = os.path.join(OUT_DIR, f"paired_runs_O_vs_OF_{DAG}_K{K_EDGE}_LightGBM_N{N_RUNS}.csv")
out_sum  = os.path.join(OUT_DIR, f"paired_ttest_summary_O_vs_OF_{DAG}_K{K_EDGE}_LightGBM_N{N_RUNS}.csv")

df_runs.to_csv(out_runs, index=False)
summary.to_csv(out_sum, index=False)

log("[DONE] saved:")
print(out_runs)
print(out_sum)